# FleetAI - GRPO Training Notebook

Train an AI inspector agent using GRPO (Group Relative Policy Optimization) with Unsloth and TRL.

## Setup

Run the cells below to install dependencies and set up the environment.

In [ ]:
#@title 1. Install Dependencies { display-mode: "form" }

# Install Unsloth (from PyPI, includes trl, transformers, datasets, etc.)
!pip install unsloth unsloth-zoo

# Install FleetAI environment
!pip install openenv-core pydantic

print('Dependencies installed!')

In [ ]:
#@title 2. GPU Check { display-mode: "form" }
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU not available. Enable GPU in Runtime > Change runtime type.")

## Clone FleetAI Repository

In [ ]:
#@title 3. Clone FleetAI { display-mode: "form" }

# Option A: Clone from your HF Space (replace YOUR_USERNAME)
# !git clone https://huggingface.co/spaces/YOUR_USERNAME/fleet-ai

# Option B: Upload your local files using the file browser on the left

# For this demo, we'll create the environment inline
import os
if not os.path.exists('environment'):
    print("Please upload the environment/ folder or clone from HF Space")
    print("Required files: environment/*.py, train.py")

In [ ]:
#@title 4. Configuration { display-mode: "form" }

# Model settings
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct" #@param ["meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3"]
OUTPUT_DIR = "./fleet_ai_model"

# Training settings
NUM_EPISODES = 200 #@param {type:"slider", min:50, max:500, step:50}
NUM_EPOCHS = 1 #@param {type:"slider", min:1, max:3, step:1}
LEARNING_RATE = 5e-5 #@param {type:"number"}
BATCH_SIZE = 2 #@param {type:"slider", min:1, max:8, step:1}

# Curriculum settings
USE_CURRICULUM = True #@param {type:"boolean"}

# Logging
REPORT_TO = "tensorboard" #@param ["tensorboard", "wandb"]

print(f"Model: {MODEL_NAME}")
print(f"Episodes: {NUM_EPISODES}")
print(f"Curriculum: {USE_CURRICULUM}")

## Generate Training Data

In [ ]:
#@title 5. Generate Training Data { display-mode: "form" }

import sys
sys.path.insert(0, '.')

from train import generate_training_data, CURRICULUM

print(f"Curriculum stages: {len(CURRICULUM)}")
for stage in CURRICULUM:
    print(f"  {stage['name']}: {stage['episodes']} episodes ({stage['description']})")

print("\nGenerating training data...")
data_path = generate_training_data(
    num_episodes=NUM_EPISODES,
    curriculum=USE_CURRICULUM
)
print(f"Training data saved to {data_path}")

## Evaluate Baseline (Heuristic Inspector)

In [ ]:
#@title 6. Evaluate Baseline { display-mode: "form" }

from train import evaluate_baseline
import json

print("Evaluating heuristic inspector baseline...")
baseline_results = evaluate_baseline(num_tickets=10)

print("\n=== BASELINE RESULTS ===")
for task, result in baseline_results.items():
    print(f"  {task}: mean={result['mean']:.3f}")

# Save baseline results
with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)
print("\nBaseline saved to baseline_results.json")

## SFT Warm-Start Training (Optional)

In [ ]:
#@title 7. SFT Training { display-mode: "form" }

RUN_SFT = True #@param {type:"boolean"}

if RUN_SFT:
    from train import run_sft_training
    
    print("Starting SFT warm-start training...")
    run_sft_training(data_path, f"{OUTPUT_DIR}_sft", num_epochs=NUM_EPOCHS)
    print(f"SFT model saved to {OUTPUT_DIR}_sft")
else:
    print("Skipping SFT. Set RUN_SFT=True to run.")

## GRPO Reinforcement Learning Training

In [ ]:
#@title 8. GRPO Training { display-mode: "form" }

import os
os.environ["FLEET_REPORT_TO"] = REPORT_TO

from train import run_grpo_training

print("Starting GRPO RL training...")
print(f"This may take a while ({NUM_EPOCHS} epoch(s) on {NUM_EPISODES} episodes)")

run_grpo_training(data_path, OUTPUT_DIR, num_epochs=NUM_EPOCHS)

print(f"\nGRPO model saved to {OUTPUT_DIR}")

## Evaluate Trained Model

In [ ]:
#@title 9. Evaluate Trained Model { display-mode: "form" }

from train import evaluate_model

print("Evaluating trained model...")
trained_results = evaluate_model(OUTPUT_DIR, num_tickets=10)

print("\n=== TRAINED MODEL RESULTS ===")
for task, result in trained_results.items():
    print(f"  {task}: mean={result['mean']:.3f}")

# Save trained results
with open("trained_results.json", "w") as f:
    json.dump(trained_results, f, indent=2)

## Compare Baseline vs Trained

In [ ]:
#@title 10. Results Comparison { display-mode: "form" }

print("\n" + "#" * 60)
print("  BASELINE vs TRAINED COMPARISON")
print("#" * 60)
print(f"{'Task':<35} {'Baseline':>10} {'Trained':>10} {'Delta':>10}")
print("-" * 65)

tasks = ["inspection_easy", "inspection_hard", "inspection_adversarial"]
task_labels = {
    "inspection_easy": "Easy (Obvious Errors)",
    "inspection_hard": "Hard (Subtle Errors + Clean Traps)",
    "inspection_adversarial": "Adversarial (Designed to Trick)",
}

for task in tasks:
    b = baseline_results.get(task, {}).get("mean", 0)
    t = trained_results.get(task, {}).get("mean", 0)
    delta = t - b
    arrow = "+" if delta > 0 else ""
    print(f"{task_labels[task]:<35} {b:>10.3f} {t:>10.3f} {arrow}{delta:>9.3f}")

# Overall
b_all = sum(v for t in tasks if t in baseline_results for v in baseline_results[t]["scores"]) / sum(len(baseline_results[t]["scores"]) for t in tasks if t in baseline_results)
t_all = sum(v for t in tasks if t in trained_results for v in trained_results[t]["scores"]) / sum(len(trained_results[t]["scores"]) for t in tasks if t in trained_results)
print("\n" + "-" * 65)
print(f"{'OVERALL':<35} {b_all:>10.3f} {t_all:>10.3f} {('+' if t_all > b_all else '') + str(t_all - b_all):>10.3f}")
print("#" * 60)

## Generate Plots

In [ ]:
#@title 11. Generate Training Plots { display-mode: "form" }

import json
import matplotlib.pyplot as plt
import os

os.makedirs("plots", exist_ok=True)

# Load generation log
gen_log_path = os.path.join(OUTPUT_DIR, "generation_log.json")
if os.path.exists(gen_log_path):
    with open(gen_log_path) as f:
        gen_log = json.load(f)
    
    rewards_history = gen_log.get("rewards_history", [])
    
    if rewards_history:
        # Plot 1: Reward over training
        calls = [r["call"] for r in rewards_history]
        mean_rewards = [r["mean_reward"] for r in rewards_history]
        min_rewards = [r["min_reward"] for r in rewards_history]
        max_rewards = [r["max_reward"] for r in rewards_history]
        
        plt.figure(figsize=(10, 6))
        plt.plot(calls, mean_rewards, "b-", label="Mean Reward", linewidth=2)
        plt.fill_between(calls, min_rewards, max_rewards, alpha=0.2, color="blue")
        plt.xlabel("Training Step")
        plt.ylabel("Reward")
        plt.title("FleetAI Inspector Training: Reward Over Time")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig("plots/training_reward.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved: plots/training_reward.png")
        
        # Plot 2: Zero reward fraction
        zero_fracs = [r["zero_fraction"] for r in rewards_history]
        plt.figure(figsize=(10, 4))
        plt.plot(calls, zero_fracs, "r-", linewidth=2)
        plt.xlabel("Training Step")
        plt.ylabel("Zero Reward Fraction")
        plt.title("Training Health: Episodes with Zero Reward")
        plt.grid(True, alpha=0.3)
        plt.savefig("plots/zero_fraction.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved: plots/zero_fraction.png")
else:
    print("No generation log found. Plots will be created after training.")

In [ ]:
#@title 12. Generate Comparison Plot { display-mode: "form" }

import numpy as np

# Comparison bar chart
tasks_short = ["Easy", "Hard", "Adversarial"]
baseline_means = [baseline_results.get(t, {}).get("mean", 0) for t in tasks]
trained_means = [trained_results.get(t, {}).get("mean", 0) for t in tasks]

x = np.arange(len(tasks_short))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, baseline_means, width, label="Baseline (Heuristic)", color="steelblue")
bars2 = ax.bar(x + width/2, trained_means, width, label="Trained (GRPO)", color="darkorange")

ax.set_xlabel("Difficulty Level")
ax.set_ylabel("Mean Reward")
ax.set_title("FleetAI Inspector: Baseline vs Trained Performance")
ax.set_xticks(x)
ax.set_xticklabels(tasks_short)
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3, axis="y")

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)

plt.savefig("plots/baseline_vs_trained.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/baseline_vs_trained.png")

## Export and Save Results

In [ ]:
#@title 13. Save All Results { display-mode: "form" }

import shutil
from datetime import datetime

# Create results bundle
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = f"fleet_ai_results_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

# Copy plots
if os.path.exists("plots"):
    shutil.copytree("plots", os.path.join(results_dir, "plots"), dirs_exist_ok=True)

# Copy JSON results
for f in ["baseline_results.json", "trained_results.json"]:
    if os.path.exists(f):
        shutil.copy(f, results_dir)

# Copy generation log
if os.path.exists(gen_log_path):
    shutil.copy(gen_log_path, results_dir)

print(f"Results saved to: {results_dir}/")
print("\nContents:")
for f in os.listdir(results_dir):
    print(f"  {f}")

In [ ]:
#@title 14. Download Results { display-mode: "form" }

from google.colab import files
import zipfile

# Zip results
zip_path = f"{results_dir}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(results_dir):
        for file in files_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, results_dir)
            zipf.write(file_path, arcname)

files.download(zip_path)
print(f"Downloaded: {zip_path}")

## Push to HuggingFace (Optional)

In [ ]:
#@title 15. Push Model to HuggingFace { display-mode: "form" }

HF_USERNAME = "your-username" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}
MODEL_REPO = f"{HF_USERNAME}/fleet-ai-inspector"

PUSH_TO_HF = False #@param {type:"boolean"}

if PUSH_TO_HF and HF_TOKEN:
    from huggingface_hub import HfApi
    
    api = HfApi()
    api.create_repo(repo_id=MODEL_REPO, exist_ok=True, token=HF_TOKEN)
    
    # Upload model
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=MODEL_REPO,
        token=HF_TOKEN
    )
    
    # Upload plots
    api.upload_folder(
        folder_path="plots",
        path_in_repo="plots",
        repo_id=MODEL_REPO,
        token=HF_TOKEN
    )
    
    print(f"Model pushed to: https://huggingface.co/{MODEL_REPO}")
else:
    print("Set PUSH_TO_HF=True and provide HF_TOKEN to push model.")